In [4]:
#'''from ultralytics import YOLO

#model = YOLO(r"c:\Users\HP\Desktop\yolo11s.pt")   

#model.train(
 #   data=r"c:\Users\HP\Desktop\Zeyad Merged.v1i.yolov11\data.yaml",

  ###imgsz=960,
    #batch=8,
    #device=0,
    #optimizer="AdamW",
    #lr0=0.001,
    #cos_lr=True,
    #cache=True,
    #workers=8,

    #---------------
 
    #project=r"D:\OCR_Project\runs",
    #name="YOLO11s_960_Experiment1",
    #exist_ok=False
#) '''

done = True

In [5]:
import time
import os
import paddleocr
from paddleocr import PaddleOCR
import torch
import cv2  
import ultralytics
ocr = PaddleOCR( lang="ar", device="gpu:0" ,textline_orientation_batch_size=1, enable_mkldnn=False, use_angle_cls=True)
from ultralytics import YOLO
#modelID = YOLO(r'c:\Users\HP\Desktop\pc vision\OCR\OCR\detect_id_card.pt')
modelID = YOLO('detect_id_card.pt')
model = YOLO("best.pt")

#model = YOLO(r"d:\OCR_Project\runs\YOLO11s_960_Experiment1\weights\best.pt")
from ultralytics import settings
settings.update({"runs_dir": r"D:\OCR_Project\runs"})
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt


C:\Users\HP\AppData\Local\Temp\ipykernel_21904\1260930817.py:8: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR( lang="ar", device="gpu:0" ,textline_orientation_batch_size=1, enable_mkldnn=False, use_angle_cls=True)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\HP\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
d:\OCRByGPU\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manua

In [6]:
def detect_crop(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    results = modelID(img)

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            return img[y1:y2, x1:x2]       

In [7]:
#test
i = detect_crop(r"c:\Users\HP\Desktop\New folder (2)\Screenshot 2026-08-01 210640.png")
i = cv2.cvtColor(i, cv2.COLOR_GRAY2BGR)
resul = ocr.predict(i)
print(resul[0]["rec_texts"])


0: 448x640 1 front-up, 36.8ms
Speed: 6.0ms preprocess, 36.8ms inference, 12.7ms postprocess per image at shape (1, 3, 448, 640)
['حمهور ية مصَ الحربيين', 'بطاقة لحفيق الشخمية', 'نورالدين', 'محمد سمير سلام حافظ', '١ ٥ مدخل ٢  الشيخ زايد', 'الاسماعيلية ثالث  الاسماعيلية', '٢٠٠٥١١', 'JM', '٣٠٥١٠٠١', '٠٠٨٥٥', 'KT0372522']


In [8]:
def detect_objects(id_img):
    results = model.predict( source=id_img,conf=0.0,iou=True)
    return results 



In [ ]:
padding = {
    "Name1": (41, 31, 40, 50),  #ok    
    #"Name2": (60, 30, 80, 80), # -----
    "Name2": (40, 40, 40, 40), # ok

    "Add1":  (20, 20, 20, 20),
    #"Add2":  (80, 80, 80, 80), good
    #"Add2":  (10,10,10,10), # ok 
    "Add2":  (41,41,41,41),


    #"Num1":  (53, 32, 85, 68), # ok 
    #"Num1":  (20, 24, 13, 13),
    "Num1":  (60, 60, 60, 60),

    #"Num2":  (20, 20, 30, 40) # ok 
    "Num2": (20, 20, 20, 20)
}




def preprocess_image(img, class_name):

    if class_name == "Face":
        return img

    if class_name == "Name1":
        img = cv2.fastNlMeansDenoising( img, None, 11,7,20 )

    if class_name == "Num1" :
        i#mg = cv2.resize(img, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
        #img= guidedFilter(guide=img, src=img, radius=10, eps=120)
        #img = cv2.fastNlMeansDenoisingColored(img,None,h=5,hColor=12,templateWindowSize=10,searchWindowSize=20)
        img = cv2.fastNlMeansDenoising(img,None,7,7,20)

        
    if class_name == "Num2" :
        img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        img = cv2.fastNlMeansDenoising(img,None,11,7,20)


    if class_name == "Name2" :
        img = cv2.resize(img, None, fx=2, fy=1.5, interpolation=cv2.INTER_CUBIC)
        img = cv2.fastNlMeansDenoising( img, None, 11,11,20 )

    if class_name == "Add2" :
        #img = cv2.resize(img, None, fx=3.2, fy=3.2, interpolation=cv2.INTER_CUBIC) # old old old old 
        #img = cv2.fastNlMeansDenoising(img,None,5,7,20) # old old old old
        img = cv2.fastNlMeansDenoising(img,None,15,3,5)
        img = cv2.copyMakeBorder(img, 41, 41, 41, 41,cv2.BORDER_CONSTANT, value=(255, 255, 255))
        img = cv2.resize(img, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)



    if class_name == "Add1" :
        img = cv2.resize(img, None, fx=1.8, fy=1.6, interpolation=cv2.INTER_CUBIC)
        img = cv2.fastNlMeansDenoising(img,None,13,7,20)       
    





    '''if class_name == "Add2":
        metrics = compute_metrics(img)
        quality = classify_image(metrics)

        if quality == "clean":
            
            img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
            
            img = cv2.copyMakeBorder(img, 5, 5, 5, 5,
                                      cv2.BORDER_CONSTANT, value=(255, 255, 255))
        else:
            
            img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
            kernel = np.array([[ 0, -1,  0],
                                [-1, 5.6, -1],
                                [ 0, -1,  0]])
            img = cv2.filter2D(src=img, ddepth=-1, kernel=kernel)
            img = cv2.copyMakeBorder(img, 5, 5, 50, 30,
                                      cv2.BORDER_CONSTANT, value=(255, 255, 255))

        return img    '''
        


    top, bottom, left, right = padding[class_name]

    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(255,255,255))
     

    return img

In [10]:
def crop_objects(results, img, save_dir, image_id):

    boxes = results[0].boxes
    best = {}
    crops = {}

    for i in range(len(boxes)):
        cls = int(boxes.cls[i].item())
        conf = float(boxes.conf[i].item())
        box = boxes.xyxy[i].cpu().numpy()

        if cls not in best or conf > best[cls][0]:
            best[cls] = (conf, box)

    for cls, (conf, box) in best.items():

        x1, y1, x2, y2 = map(int, box)

        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        class_name = model.names[cls]
        crops[class_name] = crop

        class_dir = os.path.join(save_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        filename = f"{image_id}.jpg"

        cv2.imwrite(
            os.path.join(class_dir, filename),
            crop
        )

        print(f"{class_name} -> {filename} ({conf:.3f})")

    return crops

In [11]:
def create_dataframe(cropsPATH):

    records = {}

    for class_name in sorted(os.listdir(cropsPATH)):

        class_dir = os.path.join(cropsPATH, class_name)

        if not os.path.isdir(class_dir):
            continue

        files = sorted(
            [
                f for f in os.listdir(class_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
        )

        for file in files:

            img_path = os.path.join(class_dir, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            #img = preprocess_image(img)
            img = preprocess_image(img, class_name)

            if class_name == "Face":
                text = img_path
            else:
                result = ocr.predict(img)

                if len(result[0]["rec_texts"]) > 0:
                    text = " ".join(result[0]["rec_texts"])
                else:
                    text = ""

            image_id = os.path.splitext(file)[0]

            if image_id not in records:
                records[image_id] = {
                    "Add1": "",
                    "Add2": "",
                    "Face": "",
                    "Name1": "",
                    "Name2": "",
                    "Num1": "",
                    "Num2": ""
                }

            records[image_id][class_name] = text

    df = pd.DataFrame.from_dict(records, orient="index")
    df.index.name = "Image"
    df.reset_index(inplace=True)

    return df

In [42]:
import os

images_dir = r"c:\Users\HP\Desktop\New folder (2)"
save_dir = r"D:\OCR_Project\crops_YOLO11s"

for idx, file in enumerate(sorted(os.listdir(images_dir)), start=1):

    if not file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(images_dir, file)

    Id = detect_crop(img_path)

    if Id is None:
        print(f"ID Card not detected: {file}")
        continue

    results = detect_objects(Id)

    crop_objects(results, Id, save_dir, str(idx))

df_new = create_dataframe(save_dir)



0: 640x640 1 front-up, 19.2ms
Speed: 2.6ms preprocess, 19.2ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

0: 960x960 15 Add1s, 32 Add2s, 15 Faces, 145 Name1s, 31 Name2s, 16 Num1s, 46 Num2s, 26.8ms
Speed: 6.2ms preprocess, 26.8ms inference, 26.9ms postprocess per image at shape (1, 3, 960, 960)
Num2 -> 1.jpg (0.826)
Face -> 1.jpg (0.798)
Num1 -> 1.jpg (0.775)
Name2 -> 1.jpg (0.736)
Add1 -> 1.jpg (0.663)
Add2 -> 1.jpg (0.605)
Name1 -> 1.jpg (0.472)

0: 640x640 1 front-up, 7.3ms
Speed: 3.3ms preprocess, 7.3ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

0: 960x960 44 Add1s, 48 Add2s, 15 Faces, 101 Name1s, 57 Name2s, 22 Num1s, 13 Num2s, 14.3ms
Speed: 5.3ms preprocess, 14.3ms inference, 21.7ms postprocess per image at shape (1, 3, 960, 960)
Face -> 2.jpg (0.771)
Num2 -> 2.jpg (0.763)
Num1 -> 2.jpg (0.758)
Name2 -> 2.jpg (0.653)
Add1 -> 2.jpg (0.630)
Add2 -> 2.jpg (0.584)
Name1 -> 2.jpg (0.530)

0: 416x640 1 front-up, 21.3ms
Speed: 1.5ms preprocess,

In [43]:
df_new["Image"] = df_new["Image"].astype(int)

df_new = df_new.sort_values(by="Image")
df_new = df_new[
    [
        "Image",
        "Face",
        "Add2",
        "Add1",
        "Num2",
        "Num1",
        "Name2",
        "Name1"
    ]
]

df_new.tail(100
            )

,Image,Face,Add2,Add1,Num2,Num1,Name2,Name1
0,1,D:\OCR_Project\crops_YOLO11s\Face\1.jpg,العجوزه الجيزة,٢٧ ش السودان المهندسين,GZ2108033,٢٨٩١٠٠١٨٨٠١٤٣٦,خيرى حسن بيومى حصن عوالى,محمد
11,2,D:\OCR_Project\crops_YOLO11s\Face\2.jpg,الساحل القاهر,القضاعي ٦ ش,HT1836435,٢٨٧٠٦٠١٠١١٢٦٩٤,سليمان الصادق سليمان ناجى,مصطفى
22,3,D:\OCR_Project\crops_YOLO11s\Face\3.jpg,العمرانيه الجيزه,الصدر عمران ٤٨ش الحريةش مستشفى,GI8240770,٢٨٨٠٩٠٨٢١٠٢٤١٧,محمود شوقى السيدمحمد,احمد
28,4,D:\OCR_Project\crops_YOLO11s\Face\4.jpg,الدخيله الاسكندرية,ش محمود حسن من ش السلام الهانوفيل,GK9011487,٢٩٠٠٨١٠٠٢٠٠٣٤١,محمد فتحى محمد امين,فاطمه
29,5,D:\OCR_Project\crops_YOLO11s\Face\5.jpg,السنبلاوين الدقهلية مركز,طهواى,,٢٩٨٠٦٢٠١٢٠٣٨٨٤,شعبان كامل السيد,عفاف
30,6,D:\OCR_Project\crops_YOLO11s\Face\6.jpg,روض الفرج القاهره,٤٦ ش العروسى ش بديع طوسون,HW3117979,٢٩٦٠٩١٥١٨٠٢٧٣١,فتحى محمد احمد العدوى,احمد
31,7,D:\OCR_Project\crops_YOLO11s\Face\7.jpg,الدخيلة الاسكندرية,البيطاش خلف مطعم فهمى,JN6996062,١٨٤١٠٨١٨٨٧٠٤٠٨,نشأت محمد طه خليل,محمد
32,8,D:\OCR_Project\crops_YOLO11s\Face\8.jpg,الدخية الاسكندرية,البيطاش خلف مطعم فهمى,JN6996062,٣٠٦٠٨٢٣١٢٠١٩٣١,نشات محمد طه خليل,محمد
33,9,D:\OCR_Project\crops_YOLO11s\Face\9.jpg,سيدى جابر الاسكندر,ش ٥٠ حياة السراياسموحه,GU8641199,٢٩٣٠١٠١٠٢١٧٩٥٣,ابراهيم محمد ابراهيم,مؤمن
1,10,D:\OCR_Project\crops_YOLO11s\Face\10.jpg,مركز البلينا سوهاج,الباسكيه,HF4454209,٣٠١٠١٢٨٢٦٠٠٤١٨,صابر عبدالله ابراهيم,فارس


# ( accuracy of 34 row  by col )
______________________________________
______________________________________

# ((name1)) { 100% } -  no wrong cells
______________________________________

# ((name2)) { 93% }  - 2 wrong cells and 1 char 
______________________________________

# ((num1))  { 91% }  - 3 wrong cells
______________________________________

# ((num2))  { 97% }  - 1 null
______________________________________

# ((add1))  { 92% }  - 2 wrong cells and 2 chars 
______________________________________

# ((add2))  { 91% }  - 2 wrong cells and 5 chars